In [ ]:
from src.data_preprocessing import DataPreprocessor
from src.ip_utils import merge_with_ip_country

# Initialize preprocessor
preprocessor = DataPreprocessor()
fraud_df, ip_country_df, credit_df = preprocessor.load_data()

# Clean data
fraud_df_clean = preprocessor.clean_fraud_data()

### 1. Dependencies and Initialization

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline
import joblib

# Continue from EDA notebook
# Merge with IP country data
fraud_with_country = merge_with_ip_country(fraud_df_clean, ip_country_df)

# Initialize feature engineer
from src.feature_engineering import FeatureEngineer
fe = FeatureEngineer()

# Create all features
fraud_engineered = fe.create_time_features(fraud_with_country)
fraud_engineered = fe.create_transaction_velocity(fraud_engineered)
fraud_engineered = fe.create_purchase_patterns(fraud_engineered)
fraud_engineered = fe.create_device_features(fraud_engineered)

print(f"\nTotal features created: {len(fe.features_created)}")
print(f"Features: {fe.features_created}")

# Save engineered data
fraud_engineered.to_csv('data/processed/fraud_data_engineered.csv', index=False)
print("\nSaved engineered data to: data/processed/fraud_data_engineered.csv")

### 2. Data Transformation

In [ ]:
print("\n" + "="*50)
print("DATA TRANSFORMATION")
print("="*50)

# Separate features and target
X = fraud_engineered.drop('class', axis=1)
y = fraud_engineered['class']

# Identify column types
numerical_features = ['purchase_value', 'age', 'time_since_signup', 
                      'transactions_last_1h', 'transactions_last_24h',
                      'transactions_last_168h', 'purchase_to_user_avg_ratio',
                      'users_per_device', 'transactions_per_device']

categorical_features = ['source', 'browser', 'sex', 'country', 
                        'purchase_value_category', 'device_id']

# Note: We'll exclude 'user_id', 'signup_time', 'purchase_time', 'ip_address' 
# from modeling features as they are identifiers/timestamps

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'  # Drop columns not specified
)

### 3. Handle Class Imbalance

In [ ]:
print("\n" + "="*50)
print("HANDLING CLASS IMBALANCE")
print("="*50)

print(f"\nOriginal class distribution:")
print(f"Non-fraud (0): {(y == 0).sum()} ({(y == 0).mean()*100:.2f}%)")
print(f"Fraud (1): {(y == 1).sum()} ({(y == 1).mean()*100:.2f}%)")

# Strategy selection justification
print("\n" + "-"*50)
print("IMBALANCE HANDLING STRATEGY SELECTION")
print("-"*50)
print("""
Selected SMOTE (Synthetic Minority Over-sampling Technique) because:
1. Preserves all legitimate transactions (important for business)
2. Creates synthetic fraud samples that are realistic
3. Avoids information loss from undersampling
4. Works well with tree-based models we plan to use
""")

# Apply SMOTE to training data only (will be done during modeling)
# For now, let's demonstrate how it would work
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\nTraining set before SMOTE:")
print(f"Non-fraud: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.2f}%)")
print(f"Fraud: {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.2f}%)")

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(preprocessor.fit_transform(X_train), y_train)

print(f"\nTraining set after SMOTE:")
unique, counts = np.unique(y_train_resampled, return_counts=True)
for val, cnt in zip(unique, counts):
    print(f"Class {val}: {cnt} ({cnt/len(y_train_resampled)*100:.2f}%)")

# Save the preprocessor
joblib.dump(preprocessor, 'models/preprocessor.pkl')
print("\nSaved preprocessor to: models/preprocessor.pkl")